In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image

from myDataset import *
from ArchitectureMethods import *
from MetricMethods import *

In [2]:
import torchvision.models as models

# Loads model from .pth file
os.chdir('/user/HS401/ob00564/Documents/COM3001/JAFFE/Transfer Learning')

emotionTotal = 7

# Loads MobileNetV2 trained model
mobilenet = models.mobilenet_v2(weights = 'DEFAULT')
mobilenet.classifier = torch.nn.Linear(in_features=1280, out_features= emotionTotal)
mobilenet.load_state_dict(torch.load('JAFFE MobileNet.pth'))
mobilenet.to('cuda')
mobilenet.eval()

# Loads ResNet18 trained model
resnet18 = models.resnet18(weights = None)
resnet18.fc = nn.Sequential(nn.Linear(resnet18.fc.in_features,emotionTotal))
resnet18.load_state_dict(torch.load('JAFFE ResNet18.pth'))
resnet18.to('cuda')
resnet18.eval()

# Loads ResNet34 trained model
resnet34 = models.resnet34(weights = None)
resnet34.fc = nn.Sequential(nn.Linear(resnet34.fc.in_features,emotionTotal))
resnet34.load_state_dict(torch.load('JAFFE ResNet34.pth'))
resnet34.to('cuda')
resnet34.eval()

print('Fin')

/tmp/ipykernel_3623009/3339751110.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mobilenet.load_state_dict(torch.load('JAFFE MobileNet.pth'))
/tmp/ipykernel_3623009/33

Fin


In [3]:
from torch.utils.data import ConcatDataset, DataLoader
os.chdir('/user/HS401/ob00564/Documents/COM3001')

os.chdir('/user/HS401/ob00564/Documents/COM3001/JAFFE')
print(os.getcwd())

jaffe_dataset_tr = myDataset(directory = "DATASET/train", transform = test_transform)
jaffe_dataset_val = myDataset(directory = "DATASET/validation", transform = test_transform)
jaffe_dataset = myDataset(directory = "DATASET/test", transform = test_transform)
compiled_jaffe = ConcatDataset([jaffe_dataset_tr, jaffe_dataset_val, jaffe_dataset] )
jaffe_loader = DataLoader(compiled_jaffe, batch_size = 32, shuffle = False)

os.chdir('/user/HS401/ob00564/Documents/COM3001/KDEF')
print(os.getcwd())

kdef_dataset_tr = myDataset(directory = "DATASET/train", transform = test_transform)
kdef_dataset_val = myDataset(directory = "DATASET/validation", transform = test_transform)
kdef_dataset = myDataset(directory = "DATASET/test", transform = test_transform)
compiled_kdef = ConcatDataset([kdef_dataset_tr, kdef_dataset_val, kdef_dataset])
kdef_loader = DataLoader(compiled_kdef, batch_size = 32, shuffle = False)

os.chdir('/user/HS401/ob00564/Documents/COM3001/CK+')
print(os.getcwd())
ck_dataset_tr = myDataset(directory = "DATASET/train", transform = test_transform)
ck_dataset_val = myDataset(directory = "DATASET/validation", transform = test_transform)
ck_dataset = myDataset(directory = "DATASET/test", transform = test_transform)
compiled_ck = ConcatDataset([ck_dataset_tr, ck_dataset_val, ck_dataset])
ck_loader = DataLoader(compiled_ck, batch_size = 32, shuffle = False)

print(len(compiled_jaffe))
print(len(compiled_kdef))
print(len(compiled_ck))

device = 'cuda'
criterion = nn.CrossEntropyLoss()

/user/HS401/ob00564/Documents/COM3001/JAFFE
/user/HS401/ob00564/Documents/COM3001/KDEF
/user/HS401/ob00564/Documents/COM3001/CK+
213
2938
902


In [4]:
# Testing KDEF Generalizability on JAFFE using MobileNetV2
os.chdir('/user/HS401/ob00564/Documents/COM3001/JAFFE')

print('JAFFE SPLIT on MobileNetV2')
JAFFE_y_true, JAFFE_y_pred, JAFFE_y_score = test(mobilenet, device, criterion, jaffe_loader, 'Test')

print('JAFFE SPLIT on ResNet18')
JAFFE_y_true, JAFFE_y_pred, JAFFE_y_score = test(resnet18, device, criterion, jaffe_loader, 'Test')

print('JAFFE SPLIT on ResNet34')
JAFFE_y_true, JAFFE_y_pred, JAFFE_y_score = test(resnet34, device, criterion, jaffe_loader, 'Test')

JAFFE SPLIT on MobileNetV2
Test Loss: 1.9552, Test Accuracy: 16.43%
JAFFE SPLIT on ResNet18
Test Loss: 0.1981, Test Accuracy: 97.18%
JAFFE SPLIT on ResNet34
Test Loss: 0.1665, Test Accuracy: 95.77%


In [5]:
os.chdir('/user/HS401/ob00564/Documents/COM3001/CK+')

print('CK+ SPLIT MobileNetV2')
ck_y_true, ck_y_pred, ck_y_score = test(mobilenet, device, criterion, ck_loader, 'Test')
class_report(y_pred=ck_y_pred, y_true=ck_y_true)

print('CK+ SPLIT ResNet18')
ck_y_true, ck_y_pred, ck_y_score = test(resnet18, device, criterion, ck_loader, 'Test')
class_report(y_pred=ck_y_pred, y_true=ck_y_true)

print('CK+ SPLIT ResNet34')
ck_y_true, ck_y_pred, ck_y_score = test(resnet34, device, criterion, ck_loader, 'Test')
class_report(y_pred=ck_y_pred, y_true=ck_y_true)

CK+ SPLIT MobileNetV2
Test Loss: 1.9743, Test Accuracy: 7.87%
              precision    recall  f1-score   support

       Anger     0.0000    0.0000    0.0000        45
     Disgust     0.3333    0.0169    0.0323        59
        Fear     0.0000    0.0000    0.0000        25
   Happiness     0.0774    1.0000    0.1436        69
     Sadness     0.0000    0.0000    0.0000        28
    Surprise     0.0000    0.0000    0.0000        83
     Neurtal     1.0000    0.0017    0.0034       593

    accuracy                         0.0787       902
   macro avg     0.2015    0.1455    0.0256       902
weighted avg     0.6851    0.0787    0.0153       902

CK+ SPLIT ResNet18


/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Test Loss: 3.2625, Test Accuracy: 10.64%
              precision    recall  f1-score   support

       Anger     0.0952    0.0444    0.0606        45
     Disgust     0.1292    0.5254    0.2074        59
        Fear     0.0380    0.2800    0.0670        25
   Happiness     0.1892    0.7101    0.2988        69
     Sadness     0.0255    0.1786    0.0446        28
    Surprise     1.0000    0.0241    0.0471        83
     Neurtal     0.0000    0.0000    0.0000       593

    accuracy                         0.1064       902
   macro avg     0.2110    0.2518    0.1036       902
weighted avg     0.1215    0.1064    0.0470       902

CK+ SPLIT ResNet34


/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Test Loss: 3.4270, Test Accuracy: 15.41%
              precision    recall  f1-score   support

       Anger     0.0000    0.0000    0.0000        45
     Disgust     0.0728    0.8814    0.1345        59
        Fear     1.0000    0.0400    0.0769        25
   Happiness     0.4250    0.4928    0.4564        69
     Sadness     0.0000    0.0000    0.0000        28
    Surprise     0.4906    0.6265    0.5503        83
     Neurtal     0.0000    0.0000    0.0000       593

    accuracy                         0.1541       902
   macro avg     0.2841    0.2915    0.1740       902
weighted avg     0.1101    0.1541    0.0965       902



/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/user/HS401/ob00564/Documents/COM3001/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [6]:
os.chdir('/user/HS401/ob00564/Documents/COM3001/KDEF')

print('KDEF SPLIT MobileNetV2')
KDEF_y_true, KDEF_y_pred, KDEF_y_score = test(mobilenet, device, criterion, kdef_loader, 'Test')
class_report(y_pred=KDEF_y_pred, y_true=KDEF_y_true)

print('KDEF SPLIT ResNet18')
KDEF_y_true, KDEF_y_pred, KDEF_y_score = test(resnet18, device, criterion, kdef_loader, 'Test')
class_report(y_pred=KDEF_y_pred, y_true=KDEF_y_true)

print('KDEF SPLIT ResNet34')
KDEF_y_true, KDEF_y_pred, KDEF_y_score = test(resnet34, device, criterion, kdef_loader, 'Test')
class_report(y_pred=KDEF_y_pred, y_true=KDEF_y_true)

KDEF SPLIT MobileNetV2
Test Loss: 1.9543, Test Accuracy: 14.53%
              precision    recall  f1-score   support

       Anger     0.0000    0.0000    0.0000       420
     Disgust     0.2068    0.1881    0.1970       420
        Fear     0.1458    0.2905    0.1941       420
   Happiness     0.1389    0.0595    0.0833       420
     Sadness     0.0942    0.0310    0.0467       419
    Surprise     0.3214    0.0215    0.0403       419
     Neurtal     0.1306    0.4262    0.1999       420

    accuracy                         0.1453      2938
   macro avg     0.1482    0.1453    0.1088      2938
weighted avg     0.1482    0.1453    0.1088      2938

KDEF SPLIT ResNet18
Test Loss: 1.9410, Test Accuracy: 38.87%
              precision    recall  f1-score   support

       Anger     0.2474    0.5714    0.3453       420
     Disgust     0.3114    0.7786    0.4449       420
        Fear     0.3844    0.3048    0.3400       420
   Happiness     0.8286    0.6214    0.7102       420
     Sa